In [ ]:
%load_ext watermark


In [ ]:
import itertools as it
import os

import matplotlib as mpl
from matplotlib import pyplot as plt
from phyloframe import _auxlib as pfa
from phyloframe import legacy as pfl
from pyfonts import load_google_font
from teeplot import teeplot as tp

import pylib  # noqa: F401


In [ ]:
%watermark -diwmuv -iv


In [ ]:
teeplot_subdir = os.environ.get("NOTEBOOK_NAME", "2026-03-04-fossil-foliage")
teeplot_subdir


In [ ]:
pfa.seed_random(1)


In [ ]:
font = load_google_font("Merriweather", weight=300)
mpl.font_manager.fontManager.addfont(font.get_file())
plt.rcParams["font.family"] = font.get_name()


## Prep Data


In [ ]:
df_pure = pfl.alifestd_join_roots(
    pylib.read_parquet_with_retry("https://osf.io/download/pfvsg"),
)
df_pure


In [ ]:
df_sweep = pfl.alifestd_join_roots(
    pylib.read_parquet_with_retry("https://osf.io/download/nk69s"),
)
df_sweep


In [ ]:
dfs = []
for df in (df_pure, df_sweep):
    df["x"] = df["position"] // df["nCol"]
    df["x_"] = df["x"] / df["nRow"]
    df["y"] = df["position"] % df["nCol"]
    df["y_"] = df["y"] / df["nCol"]

    df["origin_time"] = df["dstream_rank"]

    dfs.append(df)

df_pure, df_sweep = dfs


## Plot Foliage Tree


In [ ]:
for s, regime, layout in it.product(
    (1, 3),
    ("pure", "sweep"),
    ("vertical", "horizontal", "radial"),
):
    df = {
        "pure": df_pure,
        "sweep": df_sweep,
    }[regime]
    with tp.teed(
        pylib.chloropleth.draw_ctree,
        df.pipe(
            pfl.alifestd_downsample_tips_asexual, n_downsample=5_000, seed=1
        ),
        x="x_",
        y="y_",
        cmap=pylib.cmap.bcyr.get_color,
        layout=layout,
        scatter_kws=dict(
            edgecolor="none",
            s=s,
        ),
        scatter_shuffle=1,
        tree_kws=dict(
            edge=dict(
                color="gainsboro",
                linewidth=0.5,
            ),
            margins=-0.05,
        ),
        teeplot_outattrs={"regime": regime, "s": s},
        teeplot_subdir=teeplot_subdir,
    ) as teed:
        teed.invert_xaxis()
        teed.invert_yaxis()
        teed.figure.set_size_inches(3, 1.33)


## Plot Fossil Scatterplot


In [ ]:
for regime in "pure", "sweep":
    df = {
        "pure": df_pure,
        "sweep": df_sweep,
    }[regime]
    with tp.teed(
        pylib.chloropleth.draw_cscatter,
        df.pipe(
            pfl.alifestd_downsample_tips_asexual, n_downsample=5_000, seed=1
        ).dropna(subset=["x_", "y_"]),
        x="x",
        y="y",
        cmap=pylib.cmap.bcyr.get_color,
        despine=True,
        major=100,
        minor=25,
        xmax=1170,
        ymax=755,
        scatter_kws=dict(
            alpha=0.8,
            s=5,
        ),
        teeplot_dpi=600,
        teeplot_outattrs={"cmap": "bcyr", "regime": regime},
        teeplot_subdir=teeplot_subdir,
    ) as teed:
        teed.set_aspect("equal")
        fig = teed.figure
        fig.set_size_inches(3, 2)
        fig.tight_layout()


## Combined Plots


In [ ]:
with tp.teed(
    plt.subplots,
    ncols=2,
    nrows=2,
    figsize=(6, 4),
    gridspec_kw={"height_ratios": [2, 1], "hspace": 0.1, "wspace": 0.05},
    teeplot_subdir=teeplot_subdir,
) as fig:
    fig, axs = fig

    for col, regime in enumerate(("pure", "sweep")):
        df = {
            "pure": df_pure,
            "sweep": df_sweep,
        }[regime]

        pylib.chloropleth.draw_cscatter(
            df.copy().pipe(
                pfl.alifestd_downsample_tips_asexual, n_downsample=5_000, seed=1
            ).dropna(subset=["x_", "y_"]),
            x="x",
            y="y",
            cmap=pylib.cmap.bcyr.get_color,
            despine=True,
            major=100,
            minor=25,
            scatter_kws=dict(
                alpha=0.8,
                s=3,
            ),
            xmax=1170,
            ymax=755,
            ax=axs[0, col],
        )
        axs[0, col].set_aspect("equal", anchor="S")

        pylib.chloropleth.draw_ctree(
            df.copy().pipe(
                pfl.alifestd_downsample_tips_asexual, n_downsample=5_000, seed=1
            ),
            x="x_",
            y="y_",
            cmap=pylib.cmap.bcyr.get_color,
            layout="vertical",
            scatter_kws=dict(
                edgecolor="none",
                s=1,
            ),
            scatter_shuffle=1,
            tree_kws=dict(
                edge=dict(
                    color="gainsboro",
                    linewidth=0.5,
                ),
                margins=-0.05,
            ),
            ax=axs[1, col],
        )
        axs[1, col].invert_xaxis()
        axs[1, col].invert_yaxis()
        axs[1, col].set_anchor("N")


# S=1


In [ ]:
import pymupdf

spatial_positions_path = "assets/2026-03-04-wse-spatial-phylo-positions.pdf"
spatial_template_path = "assets/2026-03-04-wse-spatial-phylo-template.pdf"

spatial_positions_doc = pymupdf.open(spatial_positions_path)
print(
    f"Positions: {len(spatial_positions_doc)} page(s), size {spatial_positions_doc[0].rect}"
)

spatial_template_doc = pymupdf.open(spatial_template_path)
print(
    f"Template:  {len(spatial_template_doc)} page(s), size {spatial_template_doc[0].rect}"
)


In [ ]:
spatial_target_colors = {
    "accede": "#ACCEDE",
    "beefed": "#BEEFED",
    "decade": "#DECADE",
    "deadbe": "#DEADBE",
}


def hex_to_rgb_float(hex_color):
    h = hex_color.lstrip("#")
    return tuple(int(h[i : i + 2], 16) / 255.0 for i in (0, 2, 4))


def find_rects_by_color(page, hex_color, tol=2 / 255):
    target = hex_to_rgb_float(hex_color)
    rects = []
    for path in page.get_drawings():
        fill = path.get("fill")
        if fill is None or len(fill) != 3:
            continue
        if all(abs(fill[i] - target[i]) < tol for i in range(3)):
            rects.append(path["rect"])
    return rects


spatial_positions_page = spatial_positions_doc[0]
spatial_color_rects = {}
for name, hex_color in spatial_target_colors.items():
    rects = find_rects_by_color(spatial_positions_page, hex_color)
    spatial_color_rects[name] = rects
    for r in rects:
        print(f"  {name} ({hex_color}): {r}")

spatial_positions_doc.close()


In [ ]:
spatial_plot_paths = {
    "accede": "cmap=bcyr+regime=pure+viz=draw-cscatter+x=x+y=y+ext=.pdf",
    "beefed": "cmap=bcyr+regime=sweep+viz=draw-cscatter+x=x+y=y+ext=.pdf",
    "decade": "layout=vertical+regime=pure+s=1+viz=draw-ctree+x=x+y=y+ext=.pdf",
    "deadbe": "layout=vertical+regime=sweep+s=1+viz=draw-ctree+x=x+y=y+ext=.pdf",
}

spatial_plot_docs = {}
for name, filename in spatial_plot_paths.items():
    path = os.path.join("teeplots", teeplot_subdir, filename)
    print(f"Loading {name} from: {path}")
    spatial_plot_docs[name] = pymupdf.open(path)

page = spatial_template_doc[0]
for name, rects in spatial_color_rects.items():
    src_doc = spatial_plot_docs[name]
    delta = {
        "accede": 11,
        "beefed": 11,
        "decade": 5,
        "deadbe": 5,
    }[name]
    for rect in rects:
        rect.y0 -= delta + 7 * name.startswith("de")
        rect.x0 -= delta
        rect.y1 += delta- 7 * name.startswith("de")
        rect.x1 += delta

        page.show_pdf_page(rect, src_doc, 0)
        print(f"  Inserted {name} at {rect}")

# set Interpolate on all raster images so PDF viewers use smooth scaling
for img in page.get_images(full=True):
    xref = img[0]
    if "/Interpolate" not in spatial_template_doc.xref_object(xref):
        spatial_template_doc.xref_set_key(xref, "Interpolate", "true")

spatial_output_destination = f"teeplots/{teeplot_subdir}/"
os.makedirs(spatial_output_destination, exist_ok=True)
spatial_output_path = os.path.join(
    spatial_output_destination,
    "wse-spatial-phylo-filled-s1.pdf",
)
spatial_template_doc.save(spatial_output_path, garbage=4, deflate=True)
spatial_template_doc.close()
for d in spatial_plot_docs.values():
    d.close()
print(f"\nSaved to {spatial_output_path}")


In [ ]:
spatial_filled_doc = pymupdf.open(spatial_output_path)
dpi = 600
mat = pymupdf.Matrix(dpi / 72, dpi / 72)
pix = spatial_filled_doc[0].get_pixmap(matrix=mat, alpha=False)
spatial_png_path = spatial_output_path.replace(".pdf", ".png")
pix.save(spatial_png_path)
spatial_filled_doc.close()
print(f"Saved {pix.width}x{pix.height} @ {dpi} DPI to {spatial_png_path}")


# S=3


In [ ]:
import pymupdf

spatial_positions_path = "assets/2026-03-04-wse-spatial-phylo-positions.pdf"
spatial_template_path = "assets/2026-03-04-wse-spatial-phylo-template.pdf"

spatial_positions_doc = pymupdf.open(spatial_positions_path)
print(
    f"Positions: {len(spatial_positions_doc)} page(s), size {spatial_positions_doc[0].rect}"
)

spatial_template_doc = pymupdf.open(spatial_template_path)
print(
    f"Template:  {len(spatial_template_doc)} page(s), size {spatial_template_doc[0].rect}"
)


In [ ]:
spatial_target_colors = {
    "accede": "#ACCEDE",
    "beefed": "#BEEFED",
    "decade": "#DECADE",
    "deadbe": "#DEADBE",
}

spatial_positions_page = spatial_positions_doc[0]
spatial_color_rects = {}
for name, hex_color in spatial_target_colors.items():
    rects = find_rects_by_color(spatial_positions_page, hex_color)
    spatial_color_rects[name] = rects
    for r in rects:
        print(f"  {name} ({hex_color}): {r}")

spatial_positions_doc.close()


In [ ]:
spatial_plot_paths = {
    "accede": "cmap=bcyr+regime=pure+viz=draw-cscatter+x=x+y=y+ext=.pdf",
    "beefed": "cmap=bcyr+regime=sweep+viz=draw-cscatter+x=x+y=y+ext=.pdf",
    "decade": "layout=vertical+regime=pure+s=3+viz=draw-ctree+x=x+y=y+ext=.pdf",
    "deadbe": "layout=vertical+regime=sweep+s=3+viz=draw-ctree+x=x+y=y+ext=.pdf",
}

spatial_plot_docs = {}
for name, filename in spatial_plot_paths.items():
    path = os.path.join("teeplots", teeplot_subdir, filename)
    print(f"Loading {name} from: {path}")
    spatial_plot_docs[name] = pymupdf.open(path)

page = spatial_template_doc[0]
for name, rects in spatial_color_rects.items():
    src_doc = spatial_plot_docs[name]
    delta = {
        "accede": 11,
        "beefed": 11,
        "decade": 5,
        "deadbe": 5,
    }[name]
    for rect in rects:
        rect.y0 -= delta + 7 * name.startswith("de")
        rect.x0 -= delta
        rect.y1 += delta- 7 * name.startswith("de")
        rect.x1 += delta

        page.show_pdf_page(rect, src_doc, 0)
        print(f"  Inserted {name} at {rect}")

# set Interpolate on all raster images so PDF viewers use smooth scaling
for img in page.get_images(full=True):
    xref = img[0]
    if "/Interpolate" not in spatial_template_doc.xref_object(xref):
        spatial_template_doc.xref_set_key(xref, "Interpolate", "true")

spatial_output_destination = f"teeplots/{teeplot_subdir}/"
os.makedirs(spatial_output_destination, exist_ok=True)
spatial_output_path = os.path.join(
    spatial_output_destination,
    "wse-spatial-phylo-filled-s3.pdf",
)
spatial_template_doc.save(spatial_output_path, garbage=4, deflate=True)
spatial_template_doc.close()
for d in spatial_plot_docs.values():
    d.close()
print(f"\nSaved to {spatial_output_path}")


In [ ]:
spatial_filled_doc = pymupdf.open(spatial_output_path)
dpi = 600
mat = pymupdf.Matrix(dpi / 72, dpi / 72)
pix = spatial_filled_doc[0].get_pixmap(matrix=mat, alpha=False)
spatial_png_path = spatial_output_path.replace(".pdf", ".png")
pix.save(spatial_png_path)
spatial_filled_doc.close()
print(f"Saved {pix.width}x{pix.height} @ {dpi} DPI to {spatial_png_path}")
